### Vérification de l'environnement
On vérifie que le notebook tourne bien avec le Python du projet (`.venv`) et la bonne version de pandas (2.3.3). Sinon, on risque d'avoir des résultats différents de ceux du correcteur.

In [20]:
import sys, pandas
print(sys.executable)
print(pandas.__version__)

/home/onyxia/work/Ensae-projet-python-2026/.venv/bin/python
2.3.3


### 1. Téléchargement de la page du calendrier de la Fed
On récupère le code HTML de la page qui liste les réunions du FOMC depuis 2021. On se présente avec un `User-Agent` pour ne pas être refusé par le site. 

In [21]:
import requests
from bs4 import BeautifulSoup

URL = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"
reponse = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (projet etudiant ENSAE)"},
                       timeout=30)
print(reponse.status_code, len(reponse.text))


200 166359


### 2. Repérage des années
On transforme le HTML avec BeautifulSoup pour pouvoir chercher dedans. On affiche les premiers titres pour vérifier que chaque année a bien son bloc (« 2025 FOMC Meetings », etc.).

In [22]:
soup = BeautifulSoup(reponse.text, "lxml")
for titre in soup.find_all("h4")[:4]:
    print(titre.get_text(strip=True), "|", titre.parent.get("class"))

2026 FOMC Meetings | ['panel-heading']
2025 FOMC Meetings | ['panel-heading']
2024 FOMC Meetings | ['panel-heading']
2023 FOMC Meetings | ['panel-heading']


### 3. Structure d'une réunion
Avant d'extraire quoi que ce soit, on regarde comment une réunion est écrite dans le HTML. Ça nous donne les noms des classes qui contiennent le mois et les jours.

In [23]:
mois = soup.find_all(class_=lambda c: c and "month" in c)
print(len(mois))
print(mois[0].parent.prettify()[:1500])

57
<div "="" class="row fomc-meeting">
 <div class="fomc-meeting__month col-xs-5 col-sm-3 col-md-2">
  <strong>
   January
  </strong>
 </div>
 <div class="fomc-meeting__date col-xs-4 col-sm-9 col-md-10 col-lg-1">
  27-28
 </div>
 <div class="col-xs-12 col-md-4 col-lg-2">
  <strong>
   Statement:
  </strong>
  <br/>
  <a href="/monetarypolicy/files/monetary20260128a1.pdf">
   PDF
  </a>
  |
  <a href="/newsevents/pressreleases/monetary20260128a.htm">
   HTML
  </a>
  <br/>
  <a href="/newsevents/pressreleases/monetary20260128a1.htm">
   Implementation Note
  </a>
 </div>
 <div class="col-xs-12 col-md-4 col-lg-3">
  <a href="/monetarypolicy/fomcpressconf20260128.htm">
   Press Conference
  </a>
  <br/>
  <br/>
  <a href="/newsevents/pressreleases/monetary20260128b.htm">
   Statement on Longer-Run Goals and Monetary Policy Strategy
  </a>
 </div>
 <div class="col-xs-12 col-md-4 col-lg-4 fomc-meeting__minutes">
  <strong>
   Minutes:
  </strong>
  <br/>
  <a href="/monetarypolicy/files/fo

### 4. Fonctions d'extraction des dates (2021-2025)
On garde le **dernier jour** de chaque réunion, car la décision est annoncée le deuxième jour. Les réunions à cheval sur deux mois (« Apr/May 30-1 ») sont gérées en prenant le dernier mois. On exclut le « notation vote », qui n'est pas une décision de taux.

L'heure de l'annonce (14h à New York) est convertie en heure de Paris. Elle tombe à 20h, parfois 19h à cause des changements d'heure décalés, donc toujours après la clôture européenne : le CAC 40 réagit le lendemain.

In [24]:
import re
import pandas as pd

MOIS = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
        "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}


def date_decision(annee, mois_txt, jours_txt):
    """Renvoie le dernier jour de la réunion (jour de l'annonce)."""
    mois_fin = mois_txt.split("/")[-1].strip()[:3].lower()
    jour_fin = int(re.findall(r"\d+", jours_txt)[-1])
    return pd.Timestamp(annee, MOIS[mois_fin], jour_fin)


def dates_fomc_recentes(html, debut=2021, fin=2025):
    """Extrait les dates de décision de la page actuelle du calendrier du FOMC."""
    soup = BeautifulSoup(html, "lxml")
    lignes = []
    for titre in soup.find_all("h4"):
        trouve = re.search(r"(\d{4}) FOMC Meetings", titre.get_text())
        if not trouve or not debut <= int(trouve.group(1)) <= fin:
            continue
        annee = int(trouve.group(1))
        panneau = titre.find_parent(class_="panel")
        for mois in panneau.find_all(class_="fomc-meeting__month"):
            jours = mois.find_next_sibling(class_="fomc-meeting__date")
            texte = jours.get_text(" ", strip=True)
            if "notation vote" in texte.lower():
                continue
            date = date_decision(annee, mois.get_text(strip=True), texte)
            heure = (date + pd.Timedelta(hours=14)).tz_localize("America/New_York")
            lignes.append({"date": date.date(), "banque": "Fed",
                           "heure_paris": heure.tz_convert("Europe/Paris").strftime("%H:%M"),
                           "type": "prevue"})
    return pd.DataFrame(lignes).sort_values("date").reset_index(drop=True)

### 5. Test de l'extraction
On lance la fonction sur la page téléchargée. On attend environ 40 réunions (8 par an pendant 5 ans).

In [25]:
fed = dates_fomc_recentes(reponse.text)
print(len(fed))
fed.head(10)

40


/tmp/ipykernel_60778/2666806210.py:31: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  heure = (date + pd.Timedelta(hours=14)).tz_localize("America/New_York")


,date,banque,heure_paris,type
0,2021-01-27,Fed,20:00,prevue
1,2021-03-17,Fed,19:00,prevue
2,2021-04-28,Fed,20:00,prevue
3,2021-06-16,Fed,20:00,prevue
4,2021-07-28,Fed,20:00,prevue
5,2021-09-22,Fed,20:00,prevue
6,2021-11-03,Fed,19:00,prevue
7,2021-12-15,Fed,20:00,prevue
8,2022-01-26,Fed,20:00,prevue
9,2022-03-16,Fed,19:00,prevue


### 6. Exploration d'une page d'archive (2020)
Pour 2000-2020, les réunions sont sur des pages d'archives, une par année. La structure est différente : chaque réunion a un titre complet du type « January 28-29 Meeting - 2020 ». On choisit 2020 parce qu'elle contient tous les cas particuliers (urgences, réunion annulée, votes par écrit).

In [26]:
URL_ARCHIVE = "https://www.federalreserve.gov/monetarypolicy/fomchistorical{annee}.htm"
ENTETES = {"User-Agent": "Mozilla/5.0 (projet etudiant ENSAE)"}

page = requests.get(URL_ARCHIVE.format(annee=2020), headers=ENTETES, timeout=30)
soup_2020 = BeautifulSoup(page.text, "lxml")
titres = [t.get_text(" ", strip=True) for t in soup_2020.find_all(["h3", "h4", "h5", "h6"])]
print(page.status_code)
[t for t in titres if re.search(r"- \d{4}$", t)]

200


['January 28-29 Meeting - 2020',
 'March 2 (unscheduled) Meeting - 2020',
 'March 15 (unscheduled) Meeting - 2020',
 'March 17-18 (cancelled) Meeting - 2020',
 'March 19 (notation vote) - 2020',
 'March 23 (notation vote) - 2020',
 'March 31 (notation vote) - 2020',
 'April 28-29 Meeting - 2020',
 'June 9-10 Meeting - 2020',
 'July 28-29 Meeting - 2020',
 'August 27 (notation vote) - 2020',
 'September 15-16 Meeting - 2020',
 'November 4-5 Meeting - 2020',
 'December 15-16 Meeting - 2020']

### 7. Lecture d'un titre d'archive
La fonction transforme un titre en date et en type de réunion :
- **prévue** pour une réunion classique ;
- **urgence** pour une réunion « unscheduled » ou une conférence téléphonique ;
- les réunions annulées et les votes par écrit sont exclus.

In [27]:
def lire_titre(titre):
    """Transforme un titre d'archive en (date, type), ou None si la réunion est à exclure."""
    texte = titre.lower()
    if "cancelled" in texte or "notation vote" in texte:
        return None
    if "meeting" not in texte and "conference call" not in texte:
        return None
    annee = int(re.search(r"(\d{4})\s*$", titre).group(1))
    partie = re.sub(r"\(.*?\)", "", titre.split(" - ")[0])
    mois = [m for m in re.findall(r"[A-Za-z]+", partie) if m[:3].lower() in MOIS]
    jour = int(re.findall(r"\d+", partie)[-1])
    date = pd.Timestamp(annee, MOIS[mois[-1][:3].lower()], jour)
    urgence = "unscheduled" in texte or "conference call" in texte
    return date, "urgence" if urgence else "prevue"

On teste 7 sur trois titres pour vérifier chaque cas.

In [28]:
for t in ["January 28-29 Meeting - 2020", "March 15 (unscheduled) Meeting - 2020",
          "March 19 (notation vote) - 2020"]:
    print(t, "→", lire_titre(t))

January 28-29 Meeting - 2020 → (Timestamp('2020-01-29 00:00:00'), 'prevue')
March 15 (unscheduled) Meeting - 2020 → (Timestamp('2020-03-15 00:00:00'), 'urgence')
March 19 (notation vote) - 2020 → None


### 8. Extraction de toutes les archives (2000-2020)
On boucle sur les 21 pages d'archives. On attend une seconde entre chaque page pour ne pas surcharger le site. Avant 2013, la Fed publiait ses décisions vers 14h15 à New York, et à 14h depuis. Pour les urgences, l'heure varie (parfois le matin, parfois un dimanche), donc on la laisse vide.

In [29]:
import time

def dates_fomc_archives(debut=2000, fin=2020):
    """Extrait les dates de décision des pages d'archives du FOMC."""
    lignes = []
    for annee in range(debut, fin + 1):
        page = requests.get(URL_ARCHIVE.format(annee=annee), headers=ENTETES, timeout=30)
        page.raise_for_status()
        soup_annee = BeautifulSoup(page.text, "lxml")
        for t in soup_annee.find_all(["h3", "h4", "h5", "h6"]):
            titre = t.get_text(" ", strip=True)
            if not re.search(r"- \d{4}$", titre):
                continue
            lu = lire_titre(titre)
            if lu is None:
                continue
            date, genre = lu
            heure_ny = "14:15" if annee < 2013 else "14:00"
            heure = pd.Timestamp(f"{date.date()} {heure_ny}").tz_localize("America/New_York")
            lignes.append({"date": date.date(), "banque": "Fed",
                           "heure_paris": heure.tz_convert("Europe/Paris").strftime("%H:%M")
                                          if genre == "prevue" else None,
                           "type": genre})
        time.sleep(1)
    return pd.DataFrame(lignes).drop_duplicates("date").sort_values("date").reset_index(drop=True)

archives = dates_fomc_archives()
print(len(archives))
archives["type"].value_counts()

195


type
prevue     168
urgence     27
Name: count, dtype: int64

### 9. Contrôles des archives
On compte les réunions prévues par année (on attend 8) et on liste les urgences. On doit retrouver les grandes dates : le 17 septembre 2001, le 22 janvier 2008 et mars 2020 (Covid). Certaines urgences n'ont pas changé les taux : on les filtrera à l'étape suivante avec les données de FRED.

In [30]:
archives["annee"] = pd.to_datetime(archives["date"]).dt.year
print(archives[archives["type"] == "prevue"].groupby("annee").size())
archives[archives["type"] == "urgence"]

annee
2000    8
2001    8
2002    8
2003    9
2004    8
2005    8
2006    8
2007    8
2008    8
2009    8
2010    8
2011    8
2012    8
2013    8
2014    8
2015    8
2016    8
2017    8
2018    8
2019    8
2020    7
dtype: int64


,date,banque,heure_paris,type,annee
8,2001-01-03,Fed,None,urgence,2001
11,2001-04-11,Fed,None,urgence,2001
12,2001-04-18,Fed,None,urgence,2001
16,2001-09-13,Fed,None,urgence,2001
17,2001-09-17,Fed,None,urgence,2001
31,2003-03-25,Fed,None,urgence,2003
32,2003-04-01,Fed,None,urgence,2003
33,2003-04-08,Fed,None,urgence,2003
34,2003-04-16,Fed,None,urgence,2003
71,2007-08-10,Fed,None,urgence,2007


**Problème :** on trouve 9 réunions prévues en 2003 et 7 en 2020, contre les 8 habituelles du FOMC.

Après vérification, 2020 compte bien 7 réunions : celle de mars a été annulée et remplacée par deux réunions d'urgence (3 et 15 mars, début du Covid). Il faut donc vérifier les dates de 2003. On y trouve effectivement un doublon : la réunion de septembre apparaît le 15 et le 16, alors qu'elle s'est terminée le 16, jour de l'annonce.

**Correction :** on applique une règle simple. Si deux réunions prévues sont à deux jours ou moins d'écart, on garde la dernière. Elle ne retire que le 15 septembre 2003, et on retrouve 8 réunions par an (sauf 2020).

In [36]:
archives[archives["annee"].isin([2003]) & (archives["type"] == "prevue")]

,date,banque,heure_paris,type,annee
29,2003-01-29,Fed,20:15,prevue,2003
30,2003-03-18,Fed,20:15,prevue,2003
35,2003-05-06,Fed,20:15,prevue,2003
36,2003-06-25,Fed,20:15,prevue,2003
37,2003-08-12,Fed,20:15,prevue,2003
39,2003-09-16,Fed,20:15,prevue,2003
40,2003-10-28,Fed,20:15,prevue,2003
41,2003-12-09,Fed,20:15,prevue,2003


In [37]:
prevues = archives[archives["type"] == "prevue"]
dates = pd.to_datetime(prevues["date"])
doublon = (dates.shift(-1) - dates) <= pd.Timedelta(days=2)
print(prevues[doublon])
archives = archives.drop(prevues.index[doublon])
print(archives[archives["type"] == "prevue"].groupby("annee").size().value_counts())

Empty DataFrame
Columns: [date, banque, heure_paris, type, annee]
Index: []
8    20
7     1
Name: count, dtype: int64


/tmp/ipykernel_60778/866471459.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  doublon = (dates.shift(-1) - dates) <= pd.Timedelta(days=2)


### 11. Fichier final des annonces de la Fed
On rassemble les archives (2000-2020) et la page récente (2021-2025) dans un seul tableau, au format commun avec la BCE : `date`, `banque`, `heure_paris`, `type`. Les urgences sont encore toutes présentes : on gardera seulement celles qui ont modifié les taux à l'étape suivante, avec FRED.

In [38]:
from pathlib import Path

toutes = pd.concat([archives.drop(columns="annee"), fed], ignore_index=True)
toutes = toutes.sort_values("date").reset_index(drop=True)

print(len(toutes))
print(toutes["type"].value_counts())
print("Doublons :", toutes["date"].duplicated().sum())

chemin = Path("..") / "data" / "annonces_fed.csv"
toutes.to_csv(chemin, index=False)
print("Écrit dans :", chemin.resolve())

234
type
prevue     207
urgence     27
Name: count, dtype: int64
Doublons : 0
Écrit dans : /home/onyxia/work/Ensae-projet-python-2026/data/annonces_fed.csv
